# Pathway ↔ Gene Relation-Wise Merge

Merges Pathway–Gene triples from PrimeKG; drops rows with missing `tail_detail_name`;
deduplicates by `(head, relation, tail)`; and saves the result.

## 0. Configuration

In [22]:
import os
import pandas as pd

BASE_DIR   = '/storage/Arushi/090526_EvoAge/kg_formation/'
PROC_DIR   = BASE_DIR + 'processed_data/'
DB_DIR     = BASE_DIR + 'data_collection/databases_for_mapping/'

OUT_PATH   = BASE_DIR + 'processed_data_relation_wise_merge/generalised/PATHWAY_GENE/ALL_PATHWAY_GENE.csv'

REQUIRED_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source',
    'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type', 'species'
]

## 1. Build Gene Lookup Dictionaries

In [2]:
ENS_2NCBI = pd.read_csv(DB_DIR + 'ENSEMBL/ENSEMBLE_ID_2_NCBI_Gene_SYM.csv')
ENS_2NCBI = ENS_2NCBI[~ENS_2NCBI['name'].isna()]
NCBI_2ENS__dict = dict(zip(ENS_2NCBI['name'], ENS_2NCBI['initial_alias']))
del ENS_2NCBI

NCBI_ALL_GENE = pd.read_csv(DB_DIR + 'ncbi/Homo_sapiens.gene_info', sep='\t')
NCBI_ALL_GENE['ENSEMBLE_ID'] = NCBI_ALL_GENE['Symbol'].map(NCBI_2ENS__dict)
NCBI_ALL_GENEname_dict   = dict(zip(NCBI_ALL_GENE['GeneID'],  NCBI_ALL_GENE['description']))
NCBI_ALL_GENEIDname_dict = dict(zip(NCBI_ALL_GENE['GeneID'],  NCBI_ALL_GENE['Symbol']))
NCBI_ALL_Symb_Desc_dict  = dict(zip(NCBI_ALL_GENE['Symbol'],  NCBI_ALL_GENE['description']))

NCBI_ALL_GENE_Syn_Dict = dict(zip(NCBI_ALL_GENE['Synonyms'], NCBI_ALL_GENE['Symbol']))
exploded_dict_NCBI_ALL_GENE_Syn_Dict = {}
for k, v in NCBI_ALL_GENE_Syn_Dict.items():
    for alias in k.split('|'):
        exploded_dict_NCBI_ALL_GENE_Syn_Dict[alias.strip()] = v

print(f"NCBI gene table: {len(NCBI_ALL_GENE):,} rows")
print(f"Synonym dict:    {len(exploded_dict_NCBI_ALL_GENE_Syn_Dict):,} entries")

NCBI gene table: 193,505 rows
Synonym dict:    69,564 entries


## 2. Load KG Sources

### 2a. PrimeKG

In [12]:
PrimeKG_Pathway_Gene = pd.read_csv(PROC_DIR + 'PrimeKG/PrimeKG_Pathway_Protein_1.csv')
PrimeKG_Pathway_Gene.columns = PrimeKG_Pathway_Gene.columns.str.lower()
# Drop rows with no tail_detail_name already in the source file
PrimeKG_Pathway_Gene = PrimeKG_Pathway_Gene[~PrimeKG_Pathway_Gene['tail_detail_name'].isna()].reset_index(drop=True)
print(f"PrimeKG: {len(PrimeKG_Pathway_Gene):,} rows | columns: {list(PrimeKG_Pathway_Gene.columns)}")

PrimeKG_Pathway_Gene['kg_type'] = 'Generalised'
PrimeKG_Pathway_Gene['kg_source'] = 'PrimeKG'
PrimeKG_Pathway_Gene['species'] = 'Homo species'
PrimeKG_Pathway_Gene.head(2)

PrimeKG: 42,569 rows | columns: ['head', 'relation', 'tail', 'head_type', 'relation_type', 'tail_type', 'kg_source', 'head_id_is', 'tail_id_is', 'head_detail_name', 'tail_alias_mapped', 'tail_detail_name', 'tail_ens']


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_alias_mapped,tail_detail_name,tail_ens,kg_type,species
0,R-HSA-114608,Pathway_Gene,A1BG,Pathway,interacts with,Gene,PrimeKG,Reactome_ID,NCBI_ID,Platelet degranulation,A1BG,alpha-1-B glycoprotein,ENSG00000121410,Generalised,Homo species
1,R-HSA-6798695,Pathway_Gene,A1BG,Pathway,interacts with,Gene,PrimeKG,Reactome_ID,NCBI_ID,Neutrophil degranulation,A1BG,alpha-1-B glycoprotein,ENSG00000121410,Generalised,Homo species


## 3. Check and Fix Duplicate Columns

In [13]:
SOURCE_DFS = [('PrimeKG_Pathway_Gene', PrimeKG_Pathway_Gene)]

for name, df in SOURCE_DFS:
    dupes = df.columns[df.columns.duplicated(keep=False)].unique().tolist()
    print(f"[{name}] {'duplicate columns: ' + str(dupes) if dupes else '✓ no duplicates'}")

[PrimeKG_Pathway_Gene] ✓ no duplicates


In [14]:
PrimeKG_Pathway_Gene = PrimeKG_Pathway_Gene.loc[:, ~PrimeKG_Pathway_Gene.columns.duplicated(keep='first')]
print("[PrimeKG_Pathway_Gene] ✓ clean")

[PrimeKG_Pathway_Gene] ✓ clean


## 4. Align Schemas and Concatenate

In [15]:
CONCAT_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source', 'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type', 'species'
]

df_list = []
for name, df in SOURCE_DFS:
    tmp = df.loc[:, ~df.columns.duplicated()].copy()
    for col in CONCAT_COLS:
        if col not in tmp.columns:
            tmp[col] = pd.NA
    df_list.append(tmp[CONCAT_COLS])

final_df = pd.concat(df_list, ignore_index=True)
print(f"Combined: {len(final_df):,} rows")
final_df.head(2)

Combined: 42,569 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,R-HSA-114608,Pathway_Gene,A1BG,Pathway,interacts with,Gene,PrimeKG,Reactome_ID,NCBI_ID,Platelet degranulation,alpha-1-B glycoprotein,Generalised,Homo species
1,R-HSA-6798695,Pathway_Gene,A1BG,Pathway,interacts with,Gene,PrimeKG,Reactome_ID,NCBI_ID,Neutrophil degranulation,alpha-1-B glycoprotein,Generalised,Homo species


## 5. Standardise Fixed-Value Columns

In [16]:
final_df['head_type']  = 'Pathway'
final_df['tail_type']  = 'Gene'
final_df['relation']   = 'Pathway_Gene'
final_df['head_id_is'] = 'Reactome_ID'
final_df['tail_id_is'] = 'NCBI_ID'

print("Unique relation:",  set(final_df['relation']))
print("Unique kg_source:", set(final_df['kg_source']))

Unique relation: {'Pathway_Gene'}
Unique kg_source: {'PrimeKG'}


## 6. Deduplicate and Add Schema Columns

In [17]:
GROUP_COLS = ['head', 'relation', 'tail']

def merge_sources(x):
    return '::'.join(sorted(set(x.dropna())))

final_df_dedup = final_df.groupby(GROUP_COLS, as_index=False).agg({
    'head_type':        'first',
    'relation_type':    'first',
    'tail_type':        'first',
    'kg_source':         merge_sources,
    'head_id_is':       'first',
    'tail_id_is':       'first',
    'head_detail_name': 'first',
    'tail_detail_name': 'first',
    
    'kg_type': merge_sources,
    'species': 'first',
})
print(f"After deduplication: {len(final_df_dedup):,} rows")

final_df_dedup = final_df_dedup[REQUIRED_COLS]
final_df_dedup.head()

After deduplication: 40,964 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,R-HSA-1059683,Pathway_Gene,CBL,Pathway,interacts with,Gene,PrimeKG,Reactome_ID,NCBI_ID,Interleukin-6 signaling,Cbl proto-oncogene,Generalised,Homo species
1,R-HSA-1059683,Pathway_Gene,IL6,Pathway,interacts with,Gene,PrimeKG,Reactome_ID,NCBI_ID,Interleukin-6 signaling,interleukin 6,Generalised,Homo species
2,R-HSA-1059683,Pathway_Gene,IL6R,Pathway,interacts with,Gene,PrimeKG,Reactome_ID,NCBI_ID,Interleukin-6 signaling,interleukin 6 receptor,Generalised,Homo species
3,R-HSA-1059683,Pathway_Gene,IL6ST,Pathway,interacts with,Gene,PrimeKG,Reactome_ID,NCBI_ID,Interleukin-6 signaling,interleukin 6 cytokine family signal transducer,Generalised,Homo species
4,R-HSA-1059683,Pathway_Gene,JAK1,Pathway,interacts with,Gene,PrimeKG,Reactome_ID,NCBI_ID,Interleukin-6 signaling,Janus kinase 1,Generalised,Homo species


## 7. QC — NaN Check

In [18]:
def nan_summary(df):
    true_nan   = df.isna().sum()
    string_nan = df.apply(lambda c: c.astype(str).str.upper().eq('NAN').sum())
    return pd.DataFrame({
        'NaN_count':          true_nan,
        "'NAN'_string_count": string_nan,
        'Total_NaN_like':     true_nan + string_nan
    })

nan_summary(final_df_dedup)

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,0,0,0
tail_type,0,0,0
kg_source,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0
head_detail_name,0,0,0


## 8. Save Output

In [23]:
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
final_df_dedup.to_csv(OUT_PATH, index=False)
print(f"Saved {len(final_df_dedup):,} rows → {OUT_PATH}")

Saved 40,964 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/PATHWAY_GENE/ALL_PATHWAY_GENE.csv
